# Demo Chương 5: Phân tích Rò rỉ Mạng qua Mô hình Học máy

**Môn học:** An toàn và Bảo mật Hệ thống Thông tin
**Trọng tâm Demo:** Xây dựng mô hình phân tích lưu lượng mạng để phát hiện Rò rỉ Dữ liệu (Data Leakage)

Trong bài thực hành này, chúng ta tập trung phân tích các luồng kết nối (network flow) nhằm phát hiện kịch bản nội gián (insider threat) - một nhân viên cố tình truyền dữ liệu ra ngoài mạng vượt thẩm quyền ngoài giờ làm việc.

## 1. Sinh dữ liệu giả lập (Data Generation)

Tạo 5,000 bản ghi dữ liệu mạng dạng flow với tỉ lệ 85% bình thường (Normal) và 15% rò rỉ dữ liệu (Leakage).

In [1]:
import sys; sys.path.append('src')
from generate_data import generate_dataset

df = generate_dataset()
df.head()

[*] Sinh 88000 bản ghi Normal và 12000 bản ghi Leakage...
[✓] Đã lưu dataset tại: /home/adc300/src/ptit/atbmtt/data/network_logs.csv
    Tổng bản ghi : 100000
    Normal (0)   : 88000 (88.0%)
    Leakage (1)  : 12000 (12.0%)

[*] 5 bản ghi đầu tiên:
          timestamp        src_ip        dst_ip  dst_port protocol  bytes_sent  bytes_recv  duration  hour_of_day  is_external_dst  label
2025-11-07 14:18:19 192.168.2.104 192.168.2.224        53      UDP      104.08      213.77     32.05           14                0      0
2025-10-11 14:03:25  172.16.0.163   1.1.1.1.251      9001      UDP        1.22      174.23     13.87           14                1      0
2025-11-12 13:14:45    10.0.1.194   172.18.0.20       110      TCP       24.40      183.40     60.32           13                0      0
2025-10-01 19:17:55   192.168.1.5 192.168.2.148     11211      TCP    17681.88       52.91      1.56           19                0      0
2025-11-10 04:50:08  172.18.0.165  104.16.85.34      8080   

,timestamp,src_ip,dst_ip,dst_port,protocol,bytes_sent,bytes_recv,duration,hour_of_day,is_external_dst,label
0,2025-11-07 14:18:19,192.168.2.104,192.168.2.224,53,UDP,104.08,213.77,32.05,14,0,0
1,2025-10-11 14:03:25,172.16.0.163,1.1.1.1.251,9001,UDP,1.22,174.23,13.87,14,1,0
2,2025-11-12 13:14:45,10.0.1.194,172.18.0.20,110,TCP,24.40,183.40,60.32,13,0,0
3,2025-10-01 19:17:55,192.168.1.5,192.168.2.148,11211,TCP,17681.88,52.91,1.56,19,0,0
4,2025-11-10 04:50:08,172.18.0.165,104.16.85.34,8080,TCP,460652.75,0.92,815.77,4,1,1


## 2. Tiền xử lý dữ liệu (Preprocessing)

Quy trình tiền xử lý bao gồm:
- Làm sạch dữ liệu và loại bỏ các cột không sử dụng (IP, Timestamp).
- Mã hóa các biến phân loại (`protocol`).
- Thay đổi đặc trưng dẫn xuất: `upload_download_ratio = bytes_sent / bytes_recv`.
- Chuẩn hóa các biến liên tục MinMaxScaler (khoảng [0,1]).
- Chia tập huấn luyện/kiểm tra 80:20.

In [2]:
from preprocess import run_preprocessing_pipeline

X_train, X_test, y_train, y_test, scaler, le = run_preprocessing_pipeline()
print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)
X_train.head()

  QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU
[*] Đọc dữ liệu: 100000 bản ghi, 11 cột

── Bước 1: Làm sạch dữ liệu ──
  [✓] Không có giá trị thiếu
  [✓] Loại bỏ 0 bản ghi trùng lặp → còn 100000 bản ghi
  [✓] Loại bỏ các cột: ['timestamp', 'src_ip', 'dst_ip']

── Bước 2: Mã hóa biến phân loại ──
  [✓] Label Encoding protocol: {'ICMP': np.int64(0), 'TCP': np.int64(1), 'UDP': np.int64(2)}

── Bước 3: Feature Engineering ──
  [✓] Tạo upload_download_ratio = bytes_sent / bytes_recv
      Min: 0.0000, Max: 75973237.0000, Mean: 21152.0086

── Chia dữ liệu (80/20 Stratified) ──
  [✓] Train: 80000 bản ghi (Normal: 70400, Leakage: 9600)
  [✓] Test : 20000 bản ghi (Normal: 17600, Leakage: 2400)

── Bước 4: Chuẩn hóa dữ liệu số (MinMaxScaler) ──
  [✓] Chuẩn hóa cột: ['bytes_sent', 'bytes_recv', 'duration']
      (fit trên train → transform trên cả train & test để tránh data leakage)

  [✓] TIỀN XỬ LÝ HOÀN TẤT
  Đặc trưng sử dụng (8): ['bytes_sent', 'bytes_recv', 'duration', 'hour_of_day', 'is_external_dst', 'dst

,bytes_sent,bytes_recv,duration,hour_of_day,is_external_dst,dst_port,protocol,upload_download_ratio
52894,0.000045,0.496310,0.903492,9,1,11211,2,0.0004
18045,0.001699,0.001486,0.020471,15,0,21,1,4.5739
67618,0.000155,0.001862,0.018540,10,1,53,2,0.3330
72886,0.000154,0.002292,0.018665,13,0,53,2,0.2693
21171,0.269238,0.000024,0.353438,0,1,443,1,45809.8447


## 3. Huấn luyện Mô hình

- **Logistic Regression (Baseline):** Mô hình phân loại tuyến tính cơ bản.
- **Random Forest (Chính):** Mô hình Decision Tree Ensemble.
- **Isolation Forest:** Mô hình phát hiện bất thường Unsupervised (Không giám sát - chỉ sử dụng nhãn Normal khi train).

In [3]:
from train_models import train_all_models

models = train_all_models(X_train, y_train)

  HUẤN LUYỆN MÔ HÌNH
  Tập train: 80000 bản ghi, 8 đặc trưng

┌─────────────────────────────────────────────────┐
│  MÔ HÌNH 1: LOGISTIC REGRESSION (BASELINE)      │
└─────────────────────────────────────────────────┘
  Cấu hình: solver='lbfgs', max_iter=1000, class_weight='balanced'


/home/adc300/src/ptit/atbmtt/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  [✓] Huấn luyện hoàn tất

  Hệ số mô hình (|lớn| = quan trọng):
    - bytes_recv                -17.7735  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    - bytes_sent                -9.8046  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    + duration                  +1.8886  ▓▓▓▓▓▓▓▓▓
    + is_external_dst           +1.8151  ▓▓▓▓▓▓▓▓▓
    - protocol                  -0.3764  ▓
    + hour_of_day               +0.0243  
    + upload_download_ratio     +0.0068  
    - dst_port                  -0.0001  

┌─────────────────────────────────────────────────┐
│  MÔ HÌNH 2: RANDOM FOREST (MÔ HÌNH CHÍNH)       │
└─────────────────────────────────────────────────┘
  Cấu hình: n_estimators=100, max_depth=10, class_weight='balanced'
  [✓] Huấn luyện hoàn tất

  Feature Importance:
    #1 upload_download_ratio     0.2902  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    #2 bytes_recv                0.2688  ▓▓▓▓▓▓▓▓▓▓▓▓▓
    #3 bytes_sent                0.2086  ▓▓▓▓▓▓▓▓▓▓
    #

## 4. Đánh giá Kết quả (Evaluation)

Sử dụng các chỉ số: **Accuracy**, **Precision**, **Recall**, **F1-Score**, **FPR** và biểu đồ **Confusion Matrix**.

In [4]:
from evaluate import evaluate_all
import matplotlib.pyplot as plt
%matplotlib inline

results = evaluate_all(models, X_test, y_test, feature_names=list(X_test.columns))

  ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST
  Tập test: 20000 bản ghi (Normal: 17600, Leakage: 2400)

  ── Logistic Regression ──
    Accuracy  : 97.4%
    Precision : 89.1%
    Recall    : 88.9%
    F1-Score  : 89.0%
    FPR       : 1.5%
    (TP=2134, TN=17339, FP=261, FN=266)

  ── Random Forest ──
    Accuracy  : 99.6%
    Precision : 97.4%
    Recall    : 99.2%
    F1-Score  : 98.3%
    FPR       : 0.4%
    (TP=2380, TN=17537, FP=63, FN=20)

  ── Isolation Forest ──
    Accuracy  : 85.8%
    Precision : 45.3%
    Recall    : 90.2%
    F1-Score  : 60.3%
    FPR       : 14.8%
    (TP=2165, TN=14988, FP=2612, FN=235)

  BẢNG TỔNG HỢP KẾT QUẢ
            Mô hình Accuracy Precision Recall F1-Score   FPR
Logistic Regression    97.4%     89.1%  88.9%    89.0%  1.5%
      Random Forest    99.6%     97.4%  99.2%    98.3%  0.4%
   Isolation Forest    85.8%     45.3%  90.2%    60.3% 14.8%

── Tạo biểu đồ ──
  [✓] Lưu confusion matrices → /home/adc300/src/ptit/atbmtt/outputs/confusion_matrices.png
  [✓] 

## 5. Kết luận \& Trực quan hóa

Các biểu đồ kết quả trực quan được xuất ra thư mục `outputs/`:

### So sánh Metrics
![](outputs/metrics_comparison.png)

### Confusion Matrices
![](outputs/confusion_matrices.png)

### ROC Curves
![](outputs/roc_curves.png)

### Feature Importance (Random Forest)
![](outputs/feature_importance.png)

- **Random Forest** cho kết quả tốt nhất khi có dữ liệu gán nhãn, nhận thức rõ nhất các đặc trưng `bytes_sent` và `upload_download_ratio`.
- Mô hình phát hiện rò rỉ dữ liệu nhận biệt rõ được hành vi truyền dữ liệu trái phép thông qua các đặc trưng ngữ cảnh (cổng giao tiếp, giờ giấc, tỉ lệ truy xuất ngoài).